# Airbnb Intelligence — Model 01: Accommodation Recommendation

![Airbnb Intelligence](../../assets/airbnb-intelligence-logo.svg)


## Objective
Recommend properties from the supplied Kathmandu property dataset based on user preferences.

## Method
A content-based recommender using TF-IDF text similarity over amenities and descriptions, combined with exact categorical preference matches and budget compatibility. Guest capacity, minimum bedrooms, and maximum budget are hard filters.

**Data provenance:** the supplied properties file labels its source as `Synthetic Sample Data`. The model is a working prototype on synthetic sample records, not a model trained on verified live Airbnb listings.

## Why content-based filtering?
There is no verified user-item interaction history in the supplied files, so collaborative filtering cannot currently be trained responsibly.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "processed").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project directory.")

sys.path.insert(0, str(PROJECT_ROOT))
from src.advanced_recommender import AdvancedPropertyRecommender

DATA = PROJECT_ROOT / "data" / "processed" / "properties_clean.csv"
properties = pd.read_csv(DATA)
model = AdvancedPropertyRecommender(properties)
print("Eligible published properties:", len(model.properties))


## Example request

In [ ]:
recommendations = model.recommend_personalized(
    {
        "neighbourhood": "Thamel",
        "guests": 2,
        "max_price": 5000,
        "amenities": ["Wi-Fi", "Kitchen", "Air conditioning"],
    },
    number=10,
)
display(recommendations)


## Dashboard model artifact

This notebook uses the recommendation engine directly. The dashboard's separate serialized model is rebuilt by `src/train_recommendation_model.py`.


In [ ]:
dashboard_model_path = PROJECT_ROOT / "models" / "property_recommender.joblib"
print("Dashboard model exists:", dashboard_model_path.is_file())
print("Dashboard model path:", dashboard_model_path)


## Interpretation and limitations

- `relevance_score` is a ranking score, not a probability or accuracy percentage.
- Exact neighbourhood/property/room-type preferences contribute to ranking.
- Amenity and description similarity is measured using TF-IDF cosine similarity.
- Guest count, minimum bedrooms, and maximum budget are hard constraints.
- Ratings and review counts are used only as tie-breakers, not as verified independent quality labels.
- No ground-truth user preference or click/booking data exists, so precision@k, recall@k, NDCG, or recommendation accuracy cannot yet be reported.
- Since the dataset is synthetic, do not present results as real marketplace recommendations.
